# OOHScout — F4 Adaptation: Data Provenance Sidecars

**Current scope:** F4 — write a `.source.yaml` sidecar next to every cached dataset in `backend/data/processed/`.

**Why:** so every number OOHScout ever prints traces back to a named, licensed source. This is the paperwork that makes a paid deliverable defensible in front of a Waco billboard attorney.

**What this notebook does (end-to-end):**
1. Defines a Python dictionary schema for provenance records.
2. Writes a `.source.yaml` next to the McLennan boundary and the IH-35 centerline (both from F1/F2 caches).
3. Reads the YAMLs back to prove they're valid.
4. Runs an audit: every `.gpkg` in `processed/` must have a matching sidecar.

## 1. Imports and paths

We need `pyyaml` to write YAML cleanly (Python doesn't ship with a YAML library). It's already installed via `uv` since `pyyaml` is a common transitive dependency.

In [1]:
from datetime import date
from pathlib import Path

import yaml

working_dir = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in [working_dir, *working_dir.parents] if (p / 'pyproject.toml').exists()),
    None,
)
assert REPO_ROOT is not None, 'Run this notebook from inside the repository.'

DATA_DIR = REPO_ROOT / 'backend' / 'data' / 'processed'
print(f'DATA_DIR: {DATA_DIR}')
print(f'yaml library version: {yaml.__version__}')

DATA_DIR: C:\Users\nguye\Documents\billboardAI\backend\data\processed
yaml library version: 6.0.3


## 2. The provenance schema

Every sidecar answers the same 8 questions. Locking the schema now means every future ingestion function fills in the same fields — no ad-hoc metadata drift.

The 8 fields:

| Field | Purpose |
|---|---|
| `source_name` | Human-readable data source name |
| `source_url` | Where you actually fetched the bytes |
| `retrieval_date` | ISO date you downloaded it |
| `retrieval_method` | REST query, Overpass, download link, etc. |
| `record_count` | How many rows / features you got |
| `license` | Full license name (e.g. "OpenStreetMap ODbL") |
| `commercial_use_allowed` | `true` / `false` / `"requires_confirmation"` |
| `redistribution_allowed` | `true` / `false` / `"requires_confirmation"` |

Extra optional fields: `freshness`, `authority`, `verified_by`, `notes`.

In [2]:
def make_source_record(
    source_name,
    source_url,
    record_count,
    license_,
    commercial_use,
    redistribution,
    retrieval_method='HTTPS download',
    authority=None,
    notes=None,
):
    """Return a dict ready to serialize as a source.yaml sidecar."""
    return {
        'source_name': source_name,
        'source_url': source_url,
        'retrieval_date': str(date.today()),
        'retrieval_method': retrieval_method,
        'record_count': int(record_count),
        'license': license_,
        'commercial_use_allowed': commercial_use,
        'redistribution_allowed': redistribution,
        'authority': authority,
        'notes': notes,
    }

# Quick sanity check
example = make_source_record(
    source_name='Test',
    source_url='https://example.com',
    record_count=42,
    license_='CC-BY 4.0',
    commercial_use=True,
    redistribution=True,
)
print(yaml.safe_dump(example, sort_keys=False))

source_name: Test
source_url: https://example.com
retrieval_date: '2026-08-30'
retrieval_method: HTTPS download
record_count: 42
license: CC-BY 4.0
commercial_use_allowed: true
redistribution_allowed: true
authority: null
notes: null



## 3. Write a sidecar for the McLennan boundary (F1 output)

F1's file: `mclennan_county_study_area.gpkg`. Source: OSM Nominatim.

OSM data is licensed under the **Open Database License (ODbL)** — commercial use is allowed as long as we attribute OpenStreetMap contributors. Redistribution is allowed under the same license.

In [3]:
import geopandas as gpd

mclennan_path = DATA_DIR / 'mclennan_county_study_area.gpkg'
mclennan_record_count = len(gpd.read_file(mclennan_path, layer='study_area'))

mclennan_record = make_source_record(
    source_name='OpenStreetMap Nominatim — McLennan County, Texas',
    source_url='https://nominatim.openstreetmap.org/search?q=McLennan+County%2C+Texas',
    record_count=mclennan_record_count,
    license_='Open Database License (ODbL) 1.0',
    commercial_use=True,
    redistribution=True,
    retrieval_method='ox.geocode_to_gdf() via OSMnx',
    authority='OpenStreetMap contributors',
    notes='Attribution required per ODbL. Store "© OpenStreetMap contributors" in any published output.',
)

mclennan_sidecar = mclennan_path.with_suffix('.source.yaml')
mclennan_sidecar.write_text(yaml.safe_dump(mclennan_record, sort_keys=False))

print(f'Wrote {mclennan_sidecar.name}')
print('---')
print(mclennan_sidecar.read_text())

Wrote mclennan_county_study_area.source.yaml
---
source_name: "OpenStreetMap Nominatim \u2014 McLennan County, Texas"
source_url: https://nominatim.openstreetmap.org/search?q=McLennan+County%2C+Texas
retrieval_date: '2026-08-30'
retrieval_method: ox.geocode_to_gdf() via OSMnx
record_count: 1
license: Open Database License (ODbL) 1.0
commercial_use_allowed: true
redistribution_allowed: true
authority: OpenStreetMap contributors
notes: "Attribution required per ODbL. Store \"\xA9 OpenStreetMap contributors\" in\
  \ any published output."



## 4. Write a sidecar for the IH-35 centerline (F2 output)

F2's file: `mclennan_ih35_centerline.gpkg`. Same source (OSM), different query — Overpass instead of Nominatim.

In [4]:
ih35_path = DATA_DIR / 'mclennan_ih35_centerline.gpkg'
ih35_record_count = len(gpd.read_file(ih35_path, layer='corridor'))

ih35_record = make_source_record(
    source_name='OpenStreetMap Overpass — IH-35 motorway inside McLennan County',
    source_url='https://overpass-api.de/api/interpreter',
    record_count=ih35_record_count,
    license_='Open Database License (ODbL) 1.0',
    commercial_use=True,
    redistribution=True,
    retrieval_method='ox.features_from_polygon(tags={"highway": ["motorway"]}) via OSMnx',
    authority='OpenStreetMap contributors',
    notes='Both directions of travel + frontage segments carrying ref="I 35". Deduped on osmid.',
)

ih35_sidecar = ih35_path.with_suffix('.source.yaml')
ih35_sidecar.write_text(yaml.safe_dump(ih35_record, sort_keys=False))

print(f'Wrote {ih35_sidecar.name}')
print('---')
print(ih35_sidecar.read_text())

Wrote mclennan_ih35_centerline.source.yaml
---
source_name: "OpenStreetMap Overpass \u2014 IH-35 motorway inside McLennan County"
source_url: https://overpass-api.de/api/interpreter
retrieval_date: '2026-08-30'
retrieval_method: 'ox.features_from_polygon(tags={"highway": ["motorway"]}) via OSMnx'
record_count: 245
license: Open Database License (ODbL) 1.0
commercial_use_allowed: true
redistribution_allowed: true
authority: OpenStreetMap contributors
notes: Both directions of travel + frontage segments carrying ref="I 35". Deduped
  on osmid.



## 5. Round-trip check — read the YAMLs back

YAML files that write successfully can still be broken (missing quotes, tabs vs spaces). Reading them back confirms they're valid.

In [5]:
for sidecar in [mclennan_sidecar, ih35_sidecar]:
    loaded = yaml.safe_load(sidecar.read_text())
    assert loaded['source_name']
    assert loaded['record_count'] > 0
    assert loaded['commercial_use_allowed'] is True
    print(f'{sidecar.name} — OK ({loaded["record_count"]} records)')

mclennan_county_study_area.source.yaml — OK (1 records)
mclennan_ih35_centerline.source.yaml — OK (245 records)


## 6. Audit — does every data file have a sidecar?

This is the check the production module will enforce automatically. Any `.gpkg` / `.geojson` missing a matching `.source.yaml` is a build failure.

In [6]:
def audit_provenance(data_dir):
    """Return list of data files that are missing a .source.yaml sidecar."""
    data_files = list(data_dir.glob('*.gpkg')) + list(data_dir.glob('*.geojson'))
    missing = []
    for f in data_files:
        sidecar = f.with_suffix('.source.yaml')
        if not sidecar.exists():
            missing.append(f.name)
    return data_files, missing

data_files, missing = audit_provenance(DATA_DIR)

print(f'Data files in {DATA_DIR.name}/: {len(data_files)}')
print(f'Missing sidecars: {len(missing)}')
for m in missing:
    print(f'  {m}')

Data files in processed/: 2
Missing sidecars: 0


## F4 completion gate

F4 ships when:

1. ✅ Sidecar schema documented (see cell 3)
2. ✅ McLennan boundary has its `.source.yaml`
3. ✅ IH-35 centerline has its `.source.yaml`
4. ⏳ Production module `backend/src/oohscout/data/provenance.py` exposes `write_source_yaml()` + `audit_provenance()`
5. ⏳ Pytest fails the build if any data file lacks a sidecar

Items 1-3 you just did in this notebook. Items 4-5 are what I'll build after you say "done learning".